In [ ]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM


# P+7
# Replace the last word of each line with GPT-2's
# seventh-highest-probability next token.

MODEL_NAME = "gpt2"

print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.eval()

poem = """
One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is.
"""


# P+7 function

def p_plus_7(line):

    # Remove any trailing whitespace from the original line
    line = line.rstrip()

    if not line:
        return line

    # Identify the final word and its punctuation
    match = re.search(r"(\S+?)([.,;:!?]*)$", line)

    if not match:
        return line

    last_word = match.group(1)
    punctuation = match.group(2)

    # Remove the final word
    prompt = line[:match.start(1)]

    prompt = prompt.rstrip()

    # print(f"PROMPT: {repr(prompt)}")

    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt")

    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # Logits for the next token
    next_token_logits = outputs.logits[0, -1, :]

    # Convert logits to probabilities
    probabilities = torch.softmax(
        next_token_logits,
        dim=-1
    )

    # Get top 10
    top_probabilities, top_token_ids = torch.topk(
        probabilities,
        10
    )

    # Select the seventh-highest probability
    seventh_token_id = top_token_ids[6]

    # find replacement word
    replacement = tokenizer.decode([seventh_token_id])

    # Reconstruct the line
    result = (
        prompt
        + ""
        + replacement
        + punctuation
    )

    return result

# Apply P+7 to every line
lines = poem.strip().splitlines()

result = []

for line in lines:
    result.append(p_plus_7(line))

# Display result
print("\n===== P+7 RESULT =====\n")

for line in result:
    print(line)

Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


===== P+7 RESULT =====

One must have a mind of her
To regard the frost and the death
Of the pine-trees crusted with oil;
And have been cold a long way
To behold the junipers shagged with white,
The spruces rough in the distant horizon
Of the January sun; and not to have
Of any misery in the sound of the sound,
In the sound of a few shots,
Which is the sound of the voice
Full of the same day
That is blowing in the same bare air
For the listener, who listens in the morning,
And, nothing himself, I
Nothing that is not there and the nothing that isn.


In [15]:
import torch
import re
import unicodedata
from transformers import AutoTokenizer, AutoModelForCausalLM


# P+Least probable word
# Replace the last word of each line with GPT-2's
# least-probable usable token.
#
# If the least-probable token contains an unknown character
# or control character, move to the next more probable token.


MODEL_NAME = "gpt2"

print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.eval()


poem = """
One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is.
"""


# ------------------------------------------------------------
# Check whether a decoded GPT-2 token contains an invalid
# character for our poem.
# ------------------------------------------------------------

def contains_invalid_character(token):
    return (
        "�" in token
        or not token.isprintable()
    )


# ------------------------------------------------------------
# P+Least probable word function
# ------------------------------------------------------------

def p_plus_wow(line):

    # Remove any trailing whitespace from the original line
    line = line.rstrip()

    if not line:
        return line

    # Identify the final word and its punctuation
    match = re.search(r"(\S+?)([.,;:!?]*)$", line)

    if not match:
        return line

    last_word = match.group(1)
    punctuation = match.group(2)

    # Remove the final word
    prompt = line[:match.start(1)]

    # Remove the space before the deleted word
    prompt = prompt.rstrip()

    # print(f"PROMPT: {repr(prompt)}")

    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt")

    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # Logits for the next token
    next_token_logits = outputs.logits[0, -1, :]

    # Convert logits to probabilities
    probabilities = torch.softmax(
        next_token_logits,
        dim=-1
    )

    # Rank the entire GPT-2 vocabulary from most
    # probable to least probable
    top_probabilities, top_token_ids = torch.topk(
        probabilities,
        50257
    )

    # Start with the least-probable token.
    #
    # 0      = most probable
    # 50246  = least probable
    token_index = 50246

    while token_index >= 0:

        token_id = top_token_ids[token_index]

        # Decode candidate token
        replacement = tokenizer.decode([token_id])

        # Reject unknown/replacement characters and
        # ALL Unicode control characters.
        if not contains_invalid_character(replacement):
            break

        # Move to the next more probable token
        token_index -= 1

    # Reconstruct the line
    result = (
        prompt
        + " "
        + replacement
        + punctuation
    )

    return result


# ------------------------------------------------------------
# Apply P+21 to every line
# ------------------------------------------------------------

lines = poem.strip().splitlines()

result = []

for line in lines:
    result.append(p_plus_wow(line))


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("\n===== P+Least probable word RESULT =====\n")

for line in result:
    print(line)

Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


===== P+Least probable word RESULT =====

One must have a mind of  サーティ
To regard the frost and the  RandomRedditor
Of the pine-trees crusted with  サーティ;
And have been cold a long Nitrome
To behold the junipers shagged with  サーティ,
The spruces rough in the distant  RandomRedditor
Of the January sun; and not to  externalTo
Of any misery in the sound of the embedreportprint,
In the sound of a few  externalToEVA,
Which is the sound of the rawdownload
Full of the same  サーティ
That is blowing in the same bare uberty
For the listener, who listens in the rawdownload,
And, nothing himself, rawdownload
Nothing that is not there and the nothing that reportprint.
